# Smart Obstacle Narrator — independent inference
Run this in a fresh Colab runtime. It loads a released custom model from the same Drive and processes uploaded media.
It does not run or import the training notebook. For continuous laptop webcam and offline speaker output, run the Windows desktop application.
The offline video warning timeline below simulates acknowledgments; it is not an audio-latency measurement.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile, sys, subprocess, os
DRIVE = Path('/content/drive/MyDrive/SmartObstacleNarrator')
ARCHIVE = DRIVE / 'source' / 'SmartObstacleNarrator.zip'
assert ARCHIVE.is_file(), f'Upload the delivered project ZIP to {ARCHIVE}'
PROJECT_PARENT = Path('/content/son_source')
PROJECT_PARENT.mkdir(exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    for item in z.infolist():
        if not (PROJECT_PARENT/item.filename).resolve().is_relative_to(PROJECT_PARENT.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(PROJECT_PARENT)
PROJECT = PROJECT_PARENT / 'SmartObstacleNarrator'
os.chdir(PROJECT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'])
sys.path.insert(0, str(PROJECT/'src'))
for name in ('datasets','models','experiments','results','notebooks','documentation'):
    (DRIVE/name).mkdir(parents=True, exist_ok=True)

## Load and verify release

In [ ]:
import json, yaml, time
from smart_obstacle_narrator.release import verify_release
from smart_obstacle_narrator.config import load_config, read_yaml
from smart_obstacle_narrator.detector import Detector
from smart_obstacle_narrator.pipeline import Pipeline
VERSION = 'v1'
weights = verify_release(DRIVE/'models'/VERSION)
cfg = load_config(PROJECT/'configs'/'app.yaml')
calibration = read_yaml(PROJECT/'configs'/'camera.yaml')  # disabled until explicitly calibrated
model = Detector(str(weights), cfg, baseline=False)
print('Loaded', weights, 'on', model.device)

## Upload an image or video
Upload one file. Images show detections immediately; temporal warning confirmation needs video frames.

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one file'
MEDIA = Path(next(iter(uploaded))).resolve()
OUTPUT = DRIVE/'results'/VERSION/f'inference_{time.time_ns()}'
OUTPUT.mkdir(parents=True)

## Run inference and save outputs

In [ ]:
import cv2
from dataclasses import asdict
from smart_obstacle_narrator.geometry import observe
from smart_obstacle_narrator.display import render
pipeline = Pipeline(cfg, calibration, camera_id=str(MEDIA))
image = cv2.imread(str(MEDIA))
if image is not None:
    detections = model.detect(image)
    raw = model.model.predict(image, device=model.device, conf=cfg['confidence'], verbose=False)[0]
    cv2.imwrite(str(OUTPUT/'annotated.jpg'), raw.plot())
    observations = [asdict(observe(d, (image.shape[1],image.shape[0]), cfg, pipeline.distance)) for d in detections]
    (OUTPUT/'observations.json').write_text(json.dumps(observations, indent=2))
    print(json.dumps(observations, indent=2))
else:
    cap = cv2.VideoCapture(str(MEDIA))
    assert cap.isOpened(), 'Cannot open media'
    fps = cap.get(cv2.CAP_PROP_FPS)
    assert 0 < fps <= 240, 'Unsupported video timing'
    events, writer, index, next_speech = [], None, 0, 0.0
    try:
        while True:
            ok, frame = cap.read()
            if not ok: break
            timestamp = index / fps
            detections = model.detect(frame)
            tracks, event = pipeline.process(detections, timestamp, (frame.shape[1],frame.shape[0]))
            if event and timestamp >= next_speech:
                events.append(asdict(event))
                pipeline.acknowledge(event, timestamp)
                # Offline scheduling approximation, NOT measured TTS duration.
                next_speech = timestamp + len(event.text.split()) / (cfg['speech_rate']/60) + cfg['speech_gap']
            canvas = render(frame, tracks, events[-1]['text'] if events else '',
                            'Offline timeline; no actual TTS', 'CUSTOM MODEL', 0)
            if writer is None:
                writer = cv2.VideoWriter(str(OUTPUT/'annotated.mp4'),cv2.VideoWriter_fourcc(*'mp4v'),fps,
                                         (canvas.shape[1],canvas.shape[0]))
                assert writer.isOpened(), 'Video writer failed'
            writer.write(canvas)
            index += 1
    finally:
        cap.release()
        if writer: writer.release()
    (OUTPUT/'warning_timeline.json').write_text(json.dumps(events, indent=2))
    print('Processed frames:', index, 'warnings:', len(events))
print('Saved outputs:', OUTPUT)

## Preview or download
The Colab audio service is not Windows SAPI. Use `python -m smart_obstacle_narrator.app --weights models/v1/best_v1.pt` locally for live TTS.

In [ ]:
from IPython.display import Image, Video, display
if (OUTPUT/'annotated.jpg').exists():
    display(Image(filename=str(OUTPUT/'annotated.jpg')))
else:
    print('Download annotated.mp4 if your browser cannot decode the OpenCV MP4 codec.')
    display(Video(str(OUTPUT/'annotated.mp4'), embed=True))